# Demo: flujo de delegación del Orquestador Multi-Agente

Este notebook muestra cómo el **Supervisor** reparte el trabajo entre el **Agente de Investigación** (búsqueda en la Vector DB local) y el **Agente de Análisis** (cálculos y sentimiento), y cómo se cierra la conversación pasando por **Síntesis** y **Validación**.

Requisitos: haber ejecutado `pip install -r requirements.txt`, tener `OPENAI_API_KEY` en `.env` y abrir el notebook con el kernel del `.venv` del proyecto. La primera búsqueda descarga el modelo de embeddings e indexa el corpus (unos segundos).

In [ ]:
import sys
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)
from IPython.display import Image, Markdown, display

from graph import build_graph
from llm import get_llm
from runner import describe_event, run_query, save_trace

graph = build_graph(get_llm())
print("Grafo compilado. Nodos:", list(graph.get_graph().nodes))

## 1. Topología

`START → supervisor`; el supervisor decide con una arista condicional (`Literal["researcher", "analyst", "synthesizer"]`). Los especialistas siempre vuelven al supervisor; la síntesis pasa por el validador, que termina o devuelve el control al supervisor.

In [ ]:
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as exc:  # draw_mermaid_png usa mermaid.ink: requiere internet
    print(f"Sin PNG ({exc}). Mermaid:\n")
    print(graph.get_graph().draw_mermaid())

## 2. Consulta que obliga a Investigador → Analista → cierre

La consulta pide un **dato documental** (días por antigüedad) y un **cálculo** (promedio y porcentaje). Cada línea impresa es un paso del grafo: la decisión del supervisor con su motivo, y el aporte de cada especialista con las fuentes que realmente recuperó.

In [ ]:
def print_event(node, update):
    print(describe_event(node, update), flush=True)

consulta_1 = (
    "Según la política de vacaciones, ¿cuántos días corresponden a empleados con 3, 7 y 12 años "
    "de antigüedad? Calcula el promedio y en qué porcentaje supera el de 12 años al de 3."
)
estado_1 = await run_query(graph, consulta_1, on_event=print_event)
display(Markdown("### Respuesta final\n\n" + estado_1["final_answer"]))

### Estado compartido: quién aportó qué

`contributions` se acumula con el reducer `operator.add`: ningún agente pisa el trabajo de otro y cada aporte guarda la instrucción que lo originó y el paso del supervisor.

In [ ]:
for c in estado_1["contributions"]:
    fuentes = ", ".join(c["sources"]) or "-"
    print(f"paso {c['iteration']:>2} | {c['agent']:<11} | fuentes: {fuentes}")
    print(f"          instrucción: {c['instruction'][:110]}")
print("\nDelegaciones:", " → ".join(d["next_agent"] for d in estado_1["decisions"]))
print("Traza guardada en", save_trace(consulta_1, estado_1))

## 3. Segunda consulta: sentimiento + variación porcentual

Aquí el analista usa `analizar_sentimiento` sobre los comentarios que trajo el investigador y `variacion_porcentual` para el eNPS.

In [ ]:
consulta_2 = (
    "Busca los comentarios abiertos de la encuesta de clima 2025 y analiza su sentimiento. "
    "Además, calcula la variación porcentual del eNPS respecto de 2024."
)
estado_2 = await run_query(graph, consulta_2, on_event=print_event)
display(Markdown("### Respuesta final\n\n" + estado_2["final_answer"]))
print("\nDelegaciones:", " → ".join(d["next_agent"] for d in estado_2["decisions"]))
print("Traza guardada en", save_trace(consulta_2, estado_2))

## 4. Condición de parada: el "supervisor infinito"

Para ver la guarda sin gastar tokens, usamos un LLM falso cuyo supervisor **siempre** pide más investigación. Después de `MAX_REVISIONS` refinamientos, la guarda en código fuerza la síntesis.

In [ ]:
from langchain_core.messages import AIMessage

from config import MAX_REVISIONS
from tests.fakes import ScriptedChatModel, decide

indeciso = build_graph(
    ScriptedChatModel(responses=[decide("researcher", "Busca un poco más", "Nunca es suficiente")]),
    researcher_llm=ScriptedChatModel(responses=[AIMessage("Sin datos nuevos.")]),
    synthesizer_llm=ScriptedChatModel(responses=[AIMessage("Respuesta con la información disponible.")]),
)
estado_3 = await run_query(indeciso, "Consulta de prueba", on_event=print_event)
print(f"\nMAX_REVISIONS = {MAX_REVISIONS} → delegaciones:", [d["next_agent"] for d in estado_3["decisions"]])